# AquaPredict AI — Exploratory Data Analysis & Model Evaluation
### SIH1692: Forecasting Future Water Requirements and Assessing Storage Capacities in Reservoirs
**Ministry of Jal Shakti / Central Water Commission**

This notebook performs exploratory analysis on the multi-decadal monthly dataset containing **3,659 records** across **14 major reservoirs** in **6 Indian states**.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Dataset Overview & Schema

In [ ]:
df = pd.read_csv('../data/water_demand_reservoir_dataset.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique States: {df['state'].unique().tolist()}")
print(f"Unique Reservoirs: {df['reservoir_name'].unique().tolist()}")
df.head()

## 2. Missing Value & Integrity Audit

In [ ]:
missing_summary = df.isnull().sum()
print("Missing Values per Column:")
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else "Zero missing values detected.")
assert len(df[df['current_storage_mcm'] < 0]) == 0, "Physical Invariant Violation: Negative storage found!"

## 3. Summary Statistics

In [ ]:
numeric_cols = [
    'population', 'rainfall_mm', 'temperature_c', 'agricultural_area_ha',
    'domestic_demand_mcm', 'industrial_demand_mcm', 'agricultural_demand_mcm',
    'total_water_demand_mcm', 'current_storage_mcm', 'storage_percentage', 'inflow_mcm'
]
df[numeric_cols].describe().round(2)

## 4. Seasonal Water Demand & Rainfall Dynamics

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='season', y='total_water_demand_mcm', palette='Blues', ax=ax[0])
ax[0].set_title('Seasonal Distribution of Total Water Demand (MCM)', fontweight='bold')
ax[0].set_ylabel('Total Water Demand (MCM)')

sns.boxplot(data=df, x='season', y='rainfall_mm', palette='YlGnBu', ax=ax[1])
ax[1].set_title('Seasonal Catchment Rainfall (mm)', fontweight='bold')
ax[1].set_ylabel('Rainfall (mm)')
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap Analysis

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', cbar=True, square=True)
plt.title('Correlation Matrix of Hydrological & Socio-Economic Variables', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Reservoir Storage Capacity Utilization by State

In [ ]:
state_storage = df.groupby('state')[['current_storage_mcm', 'reservoir_capacity_mcm']].mean()
state_storage['avg_utilization_pct'] = (state_storage['current_storage_mcm'] / state_storage['reservoir_capacity_mcm']) * 100

plt.figure(figsize=(9, 4))
state_storage['avg_utilization_pct'].sort_values().plot(kind='barh', color='#0284c7')
plt.title('Average Reservoir Capacity Utilization by State (%)', fontweight='bold')
plt.xlabel('Storage Utilization (%)')
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

## 7. Model Performance Metrics Comparison

In [ ]:
with open('../models/model_metrics.json') as f:
    metrics = json.load(f)

print(f"Selected Champion Model: {metrics['best_model']}")
print(f"Selection Criterion: {metrics['selection_criterion']}")
print(f"Test Evaluation: R² = {metrics['final_deployed_test_metrics']['R2']}, MAPE = {metrics['final_deployed_test_metrics']['MAPE']}%")

comparison_df = pd.DataFrame({
    model: {
        'Val RMSE': res['validation']['RMSE'],
        'Test R2': res['test']['R2'],
        'Test MAPE (%)': res['test']['MAPE']
    }
    for model, res in metrics['models'].items()
}).T

comparison_df